# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from build_graph_and_train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

In [3]:
df = pd.read_csv(f"../../../data/top30groups/LongLatCombined/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Create longlat feature

In [6]:
geodata = ['longitude', 'latitude']
combined_geo = df.copy()
combined_geo['longlat'] = list(zip(df['longitude'], df['latitude']))
combined_geo = combined_geo.drop(columns=geodata)

In [7]:
import ast

def to_tuple_if_needed(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val  # already a tuple

combined_geo['longlat'] = combined_geo['longlat'].apply(to_tuple_if_needed)

# Weapon type prediction

In [8]:
torch.cuda.empty_cache()


In [9]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds = []
y_trues = []
logs = []

# Default config (from previous best)
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")

    data, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
        combined_geo, label_index, continuous_col=col)

    # Train using default parameters
    best_acc, best_epoch, best_precision, best_recall, best_f1, y_pred_decoded, y_true_decoded, \
    best_precision_micro, best_recall_micro, best_f1_micro, best_precision_macro, best_recall_macro, best_f1_macro, \
    roc_auc_weighted, roc_auc_micro, roc_auc_macro, epoch_logs = train_joint(
        data, data.edge_index, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask,
        default_args, row_to_node_index, index_to_label, verbose=True)

    y_preds.append(y_pred_decoded)
    y_trues.append(y_true_decoded)
    logs.append(epoch_logs)

    # Save performance metrics
    results_path = f"Results{partition}/Results_{col}_prediction"
    with open(results_path, "w") as f:
        f.write(f"Best acc: {best_acc:.4f} at epoch {best_epoch} for {col} prediction\n")
        f.write(f"Weighted Precision: {best_precision:.4f}, Recall: {best_recall:.4f}, F1: {best_f1:.4f}\n")
        f.write(f"Macro Precision: {best_precision_macro:.4f}, Recall: {best_recall_macro:.4f}, F1: {best_f1_macro:.4f}\n")
        f.write(f"Micro Precision: {best_precision_micro:.4f}, Recall: {best_recall_micro:.4f}, F1: {best_f1_micro:.4f}\n")
        f.write(f"AUROC Weighted: {roc_auc_weighted:.4f}, Micro: {roc_auc_micro:.4f}, Macro: {roc_auc_macro:.4f}\n")

    # Save epoch timings
    log_path = f"Results{partition}/epoch_logs_{col}_prediction"
    with open(log_path, "w") as f:
        f.write('\n'.join(f"{x:.4f}" for x in epoch_logs))



Training model for weaptype1 prediction...
Epoch 01 | GCN MSE Loss: 1.1598 | NRF Loss: 3.4013 | JOINT Loss: 4.5611 | NRF Acc: 0.3722
Epoch 51 | GCN MSE Loss: 1.0917 | NRF Loss: 3.0528 | JOINT Loss: 4.1445 | NRF Acc: 0.6258
Epoch 101 | GCN MSE Loss: 1.0812 | NRF Loss: 2.8644 | JOINT Loss: 3.9456 | NRF Acc: 0.8022
Epoch 151 | GCN MSE Loss: 1.0773 | NRF Loss: 2.7155 | JOINT Loss: 3.7929 | NRF Acc: 0.8656
Epoch 201 | GCN MSE Loss: 1.0747 | NRF Loss: 2.5827 | JOINT Loss: 3.6574 | NRF Acc: 0.8780
Epoch 251 | GCN MSE Loss: 1.0720 | NRF Loss: 2.4601 | JOINT Loss: 3.5321 | NRF Acc: 0.8939
Epoch 301 | GCN MSE Loss: 1.0696 | NRF Loss: 2.3455 | JOINT Loss: 3.4151 | NRF Acc: 0.9056
Epoch 351 | GCN MSE Loss: 1.0679 | NRF Loss: 2.2379 | JOINT Loss: 3.3058 | NRF Acc: 0.9125
Epoch 401 | GCN MSE Loss: 1.0663 | NRF Loss: 2.1365 | JOINT Loss: 3.2028 | NRF Acc: 0.9166
Epoch 451 | GCN MSE Loss: 1.0644 | NRF Loss: 2.0401 | JOINT Loss: 3.1045 | NRF Acc: 0.9221
Epoch 501 | GCN MSE Loss: 1.0620 | NRF Loss: 1.9

In [14]:
best_acc

0.9321573972702026

In [10]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

'\nBest acc: 0.9205 at epoch 750 for weaptype1 prediction\nWeighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191\nMacro Precision: 0.9176, Recall: 0.9107, F1: 0.9105\nMicro Precision: 0.9205, Recall: 0.9205, F1: 0.9205\nAUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963\n\n'

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       1.00      0.98      0.99        42
        African National Congress (South Africa)       0.98      1.00      0.99        58
                                Al-Qaida in Iraq       0.97      0.83      0.89        81
        Al-Qaida in the Arabian Peninsula (AQAP)       0.98      0.90      0.94        51
                                      Al-Shabaab       1.00      1.00      1.00        58
             Basque Fatherland and Freedom (ETA)       0.98      0.90      0.94        60
                                      Boko Haram       0.98      0.95      0.96        43
  Communist Party of India - Maoist (CPI-Maoist)       1.00      0.92      0.96        24
       Corsican National Liberation Front (FLNC)       0.97      0.97      0.97        77
                       Donetsk People's Republic       1.00      1.00      1.00        59
Farabundo

In [12]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [13]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])

Saved confusion matrix for partition 100 to Results100/cm_100_weaptype1.png
